🌐 **English** | [한국어](dx_rt_02_intermediate.ko.ipynb) | [日本語](dx_rt_02_intermediate.ja.ipynb) | [中文](dx_rt_02_intermediate.zh.ipynb)

# DEEPX Tutorial 06-2 - DX-RT Intermediate

This notebook moves from CLI validation to application-level inference with the DX-RT Python and C++ APIs.

## Learning objectives

By the end of this tutorial, you will be able to:

- install the matching prebuilt <code>dx_engine</code> wheel into the Jupyter kernel,
- read tensor metadata and allocate inputs with the required shape and dtype,
- implement synchronous inference and classify a real image from preprocessing to top-5 labels,
- implement asynchronous inference with job IDs and <code>wait()</code>, and with a registered callback,
- explain buffer-lifetime rules,
- group independent samples with the batch API,
- use <code>InferenceOption.buffer_count</code> deliberately,
- build a minimal C++14 application with <code>dxrt_cxx_api.h</code>, and
- select an execution mode from latency, throughput, and ownership requirements.

All generated Python, C++, build, and report files remain under <code>&lt;dx-tutorials&gt;/notebooks/T06-DX-Runtime/workspace</code>.


> Part 2 of 3 of this tutorial. The course map and the full list of parts are in the Beginner notebook.


## Prerequisites

- Tutorial 06-1 completed.
- `uv` and `cmake` (plus `g++` from `build-essential`) for the Python environment and the C++ build.
- Network access: section 2 creates an 81 MB Python environment under `workspace/.venv-dxrt`. The `dx_engine` wheel is local, but `uv` downloads its NumPy dependency, so an offline host needs a local package index.
- Expect about 10 minutes of reading. The cells themselves run in about a minute once downloads are cached.

The next cell locates the SDK, checks these requirements, and prints a status table. Anything marked `MISSING` comes with the step that provides it.


## 1. Initialize the tutorial workspace


In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import importlib.metadata
import os
import sys
import time

import numpy as np

paths = setup_tutorial(needs=["dx_rt", "npu", "uv", "cmake"])
paths.require("dx_rt", "npu")

MODEL_DIR = WORKSPACE_DIR / "models"
REPORT_DIR = WORKSPACE_DIR / "reports"
CPP_DIR = WORKSPACE_DIR / "cpp"
CPP_BUILD_DIR = CPP_DIR / "build"

for path in (WORKSPACE_DIR, MODEL_DIR, REPORT_DIR, CPP_DIR):
    path.mkdir(parents=True, exist_ok=True)

MODEL_SOURCE = DX_WORKSPACE_DIR / "res" / "models" / "resnet50_224x224.dxnn"
if not MODEL_SOURCE.is_file():
    # Download the sample model with the DX-APP setup script (same command as in a terminal).
    !cd "{DX_APP_DIR}" && bash setup.sh --models resnet50 --no-force

MODEL_PATH = MODEL_DIR / MODEL_SOURCE.name
if MODEL_PATH.is_symlink() and not MODEL_PATH.exists():
    MODEL_PATH.unlink()   # stale link from a previous SDK location
if not MODEL_PATH.exists():
    MODEL_PATH.symlink_to(MODEL_SOURCE)

os.chdir(WORKSPACE_DIR)
print(f"Kernel Python : {sys.executable}")
print(f"Workspace     : {WORKSPACE_DIR}")
print(f"Model         : {MODEL_PATH}")


## 2. Install the Python binding in the T06 workspace

The DX-RT Debian package provides version-specific wheels under <code>/usr/share/libdxrt-bin/python</code>. A wheel built for Python 3.12 cannot be imported by a Python 3.13 kernel, so this notebook selects the wheel whose <code>cpXY</code> tag matches the running kernel.

To keep every generated file inside T06, this tutorial creates a small dedicated environment under <code>workspace/.venv-dxrt</code> and appends its site-packages directory to the kernel's module search path. Appending (not prepending) matters: the environment also contains NumPy as a dependency of the wheel, and the kernel must keep using its own NumPy. Only <code>dx_engine</code>, which the kernel does not have, is resolved from the local environment. The commands are equivalent to:

~~~bash
uv venv --python <current-jupyter-python> <T06>/workspace/.venv-dxrt
uv pip install --python <T06>/workspace/.venv-dxrt/bin/python \
  /usr/share/libdxrt-bin/python/dx_engine-<version>-cp<XY>-*.whl
~~~

This does not modify the shared Jupyter environment and does not rebuild DX-RT. Re-running the cells skips creation and installation when the local environment already contains the matching wheel.


In [ ]:
WHEEL_DIR = Path("/usr/share/libdxrt-bin/python")
python_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
matching_wheels = sorted(WHEEL_DIR.glob(f"dx_engine-*-{python_tag}-{python_tag}-*.whl"))

if not matching_wheels:
    raise FileNotFoundError(
        f"No dx_engine wheel for {python_tag} was found under {WHEEL_DIR}. "
        "Install the current libdxrt-bin package first."
    )

DX_ENGINE_WHEEL = matching_wheels[-1]
DXRT_PYTHON_ENV = WORKSPACE_DIR / ".venv-dxrt"
DXRT_PYTHON = DXRT_PYTHON_ENV / "bin" / "python"
DXRT_SITE_PACKAGES = (
    DXRT_PYTHON_ENV
    / "lib"
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
)
wheel_version = DX_ENGINE_WHEEL.name.split("-")[1]

print(f"Selected wheel : {DX_ENGINE_WHEEL}")
print(f"Local Python   : {DXRT_PYTHON}")


In [ ]:
if not DXRT_PYTHON.is_file():
    !uv venv --python "{sys.executable}" "{DXRT_PYTHON_ENV}"
else:
    print(f"Local environment already exists; skipping creation: {DXRT_PYTHON_ENV}")

installed_dx_engine = next(
    (
        distribution.version
        for distribution in importlib.metadata.distributions(
            path=[str(DXRT_SITE_PACKAGES)]
        )
        if distribution.metadata.get("Name", "").lower().replace("_", "-")
        == "dx-engine"
    ),
    None,
)

if installed_dx_engine != wheel_version:
    !uv pip install --python "{DXRT_PYTHON}" "{DX_ENGINE_WHEEL}"
else:
    print(f"dx-engine {installed_dx_engine} already matches; skipping installation.")


In [ ]:
if not DXRT_SITE_PACKAGES.is_dir():
    raise FileNotFoundError(f"Local site-packages was not created: {DXRT_SITE_PACKAGES}")
# Append, do not prepend: the kernel keeps its own NumPy; only dx_engine comes from the local environment.
if str(DXRT_SITE_PACKAGES) not in sys.path:
    sys.path.append(str(DXRT_SITE_PACKAGES))

from dx_engine import InferenceEngine, InferenceOption, __version__ as dx_engine_version

print(f"dx_engine version: {dx_engine_version}")
print(f"dx_engine source : {Path(sys.modules['dx_engine'].__file__).resolve()}")


## 3. Read tensor metadata before allocating memory

Do not infer the input shape or dtype from a model name. Ask the engine for its contract. Since v3.4, DX-RT validates NumPy dtypes to prevent undefined behavior.

The helper below uses <code>np.empty(...)</code> and then fills the buffer. This forces real writable pages to be allocated, avoiding copy-on-write zero-page problems during DMA pinning. The SDK Python examples in `<DX_ALL_SUITE_DIR>/dx-runtime/dx_rt/examples/python` use the same pattern.


In [ ]:
def make_input(tensor_info: dict, fill_value: int = 0) -> np.ndarray:
    array = np.empty(tensor_info["shape"], dtype=tensor_info["dtype"])
    array.fill(fill_value)
    return np.ascontiguousarray(array)

with InferenceEngine(str(MODEL_PATH)) as ie:
    input_info = ie.get_input_tensors_info()
    output_info = ie.get_output_tensors_info()

print("Input tensors:")
for tensor in input_info:
    print(tensor)

print("\nOutput tensors:")
for tensor in output_info:
    print(tensor)


### 3.1 Tensor ownership checklist

| Check | Safe practice |
|---|---|
| Shape | Allocate from <code>get_input_tensors_info()</code> |
| Dtype | Use the returned NumPy dtype exactly |
| Layout | Match the compiled model's visible layout |
| Contiguity | Pass C-contiguous arrays |
| Lifetime | Keep asynchronous inputs alive until completion |
| Output scope | Copy callback output only when it must outlive the callback |


## 4. Synchronous inference

<code>run()</code> blocks the calling thread until the result is ready. It is the clearest starting point for a sequential or latency-oriented application.

<img src="assets/dx-rt-execution-modes.svg" style="max-width: 1100px; width: 100%;" alt="Synchronous, asynchronous, and batch execution modes">


In [ ]:
with InferenceEngine(str(MODEL_PATH)) as ie:
    tensor_info = ie.get_input_tensors_info()[0]
    input_tensor = make_input(tensor_info)

    start = time.perf_counter()
    outputs = ie.run([input_tensor])
    elapsed_ms = (time.perf_counter() - start) * 1000

    print(f"Output tensor count : {len(outputs)}")
    for index, output in enumerate(outputs):
        print(f"output[{index}] shape={output.shape}, dtype={output.dtype}, bytes={output.nbytes}")
    print(f"Host-observed time  : {elapsed_ms:.3f} ms")
    print(f"E2E response time   : {ie.get_e2e_response_time() / 1000:.3f} ms")
    print(f"Core latency        : {ie.get_core_latency() / 1000:.3f} ms")
    print(f"NPU inference time  : {ie.get_npu_inference_time() / 1000:.3f} ms")


The four timings answer different questions:

- **Host-observed time** includes the Python call and host-side overhead around it.
- **E2E response time** is DX-RT's time for the latest request, from submission to result, including the wait for an NPU buffer and the queue wait. Since DX-RT v3.5.0, `get_latency()` returns this same value.
- **Core latency** is the E2E response time without the buffer-pool and queue waits (`get_core_latency()`, new in v3.5.0). A large gap between the two means requests are waiting in the runtime, not running slowly on the NPU.
- **NPU inference time** covers NPU execution and is only one part of end-to-end latency.

Do not label NPU time as camera-to-display latency.


### 4.1 Classify a real image

The dummy input above proves that the call works, not that the result is right. The next cells run the same `run()` on a real photo and turn the output into ImageNet labels. This is the minimum application around DX-RT: preprocessing, inference, and post-processing.

The model contract from section 3 decides the preprocessing:

| Contract item | Value for `resnet50_224x224.dxnn` | What the application does |
|---|---|---|
| Input | `[1, 224, 224, 3]`, `uint8` | Resize the image to 224x224, keep 8-bit values and HWC order, add the batch axis |
| Color order | RGB | OpenCV reads BGR, so convert BGR to RGB |
| Normalization | compiled into the DXNN | Nothing: the `uint8` input shows that the model normalizes internally |
| Output | `[1, 1000]`, `float32` class scores | Softmax, then the five highest classes |

These are the same steps as DX-APP's ResNet50 example (`SimpleResizePreprocessor` and `ClassificationPostprocessor`). The image is DX-APP's default sample for that example, and the ImageNet label list also comes from DX-APP (`src/python_example/common/utility/labels.py`).

In [ ]:
import cv2
import importlib.util
import matplotlib.pyplot as plt

# ImageNet-1k class names, loaded from the DX-APP sources without importing the rest of DX-APP.
LABELS_FILE = DX_APP_DIR / "src" / "python_example" / "common" / "utility" / "labels.py"
labels_spec = importlib.util.spec_from_file_location("dx_app_labels", LABELS_FILE)
labels_module = importlib.util.module_from_spec(labels_spec)
labels_spec.loader.exec_module(labels_module)
IMAGENET_LABELS = labels_module.get_imagenet_labels()

SAMPLE_IMAGE = DX_APP_DIR / "sample" / "img" / "sample_dog.jpg"


def preprocess(image_bgr, tensor_info):
    "Resize to the model input, BGR to RGB, keep uint8 HWC, add the batch axis."
    _, height, width, _ = tensor_info["shape"]
    resized = cv2.resize(image_bgr, (width, height), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    return np.ascontiguousarray(rgb[np.newaxis].astype(tensor_info["dtype"]))


def top_k(scores, k=5):
    "Softmax over the class scores, then the k most likely (label, probability) pairs."
    scores = scores.reshape(-1).astype(np.float64)
    probabilities = np.exp(scores - scores.max())
    probabilities /= probabilities.sum()
    best = np.argsort(probabilities)[::-1][:k]
    return [(IMAGENET_LABELS[index], float(probabilities[index])) for index in best]


image_bgr = cv2.imread(str(SAMPLE_IMAGE))
if image_bgr is None:
    raise FileNotFoundError(f"Sample image not found: {SAMPLE_IMAGE}")

with InferenceEngine(str(MODEL_PATH)) as ie:
    input_info = ie.get_input_tensors_info()[0]
    real_tensor = preprocess(image_bgr, input_info)
    real_outputs = ie.run([real_tensor])

predictions = top_k(real_outputs[0])
print(f"Image : {SAMPLE_IMAGE.name}, {image_bgr.shape[1]}x{image_bgr.shape[0]} -> input {list(real_tensor.shape)} {real_tensor.dtype}")
print(f"Output: {list(real_outputs[0].shape)} {real_outputs[0].dtype}")
for rank, (label, probability) in enumerate(predictions, start=1):
    print(f"  #{rank} {label:22s} {probability:6.1%}")

plt.figure(figsize=(6, 4.5))
plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
plt.title(", ".join(f"{label} {probability:.0%}" for label, probability in predictions[:3]))
plt.axis("off")
plt.show()

How to read the result:

- The photo shows a Shiba Inu. ImageNet-1k has no Shiba Inu class, so the probability spreads over the closest spitz-type breeds (husky, Norwegian Elkhound, Siberian Husky, and so on). A spread like this is a property of the label set, not a runtime error.
- DX-APP's `resnet50_224x224_sync.py` prints the same top-5 for this image. When your own pipeline and a reference application agree, the preprocessing and output mapping are right.
- A wrong color order or missing resize still runs without an error and only produces different labels. Check accuracy on labeled data before you trust a pipeline.

## 5. Asynchronous inference with job IDs

<code>run_async()</code> returns a job ID. <code>wait(job_id)</code> later returns the matching output. This allows the application to overlap submission, NPU execution, and other work.

<img src="assets/dx-rt-buffer-lifecycle.svg" style="max-width: 1100px; width: 100%;" alt="Asynchronous buffer ownership lifecycle">


In [ ]:
ASYNC_JOBS = 8

option = InferenceOption()
option.buffer_count = 4

with InferenceEngine(str(MODEL_PATH), option) as ie:
    tensor_info = ie.get_input_tensors_info()[0]
    async_inputs = [make_input(tensor_info, index) for index in range(ASYNC_JOBS)]

    start = time.perf_counter()
    job_ids = [ie.run_async([array]) for array in async_inputs]
    async_outputs = [ie.wait(job_id) for job_id in job_ids]
    elapsed_s = time.perf_counter() - start

print(f"Submitted job IDs : {job_ids}")
print(f"Completed outputs : {len(async_outputs)}")
print(f"Throughput         : {ASYNC_JOBS / elapsed_s:.2f} requests/s")


### 5.1 Wait versus callback

| Completion style | Strength | Main responsibility |
|---|---|---|
| <code>run_async()</code> + <code>wait(job_id)</code> | Explicit request/result matching | Store job IDs and wait from an appropriate thread |
| Registered callback | Low-latency completion handling | Keep the callback fast and thread-safe |
| <code>run()</code> | Simplest control flow | Accept a blocked calling thread |

Callback outputs are valid only inside the callback scope. If downstream work must retain them, copy the required data inside the callback and move heavy work to another queue.


The next cell uses the callback style. `register_callback()` sets a function that DX-RT calls on its own worker thread when each `run_async()` job finishes. `user_arg` travels with the job, so the callback knows which request finished, for example a frame number. The cell:

1. registers the callback,
2. submits eight jobs with the real image from section 4.1, each with its index as `user_arg`,
3. waits for an event that the callback sets after the last job, and
4. clears the callback with `register_callback(None)`.

The callback takes a lock before it changes shared data, keeps only the small results it needs (the top-1 label), and returns `0`. Heavy post-processing belongs on another thread or queue, so the runtime's worker thread is not blocked. This is the pattern that DX-APP's `AsyncRunner` builds on.

In [ ]:
import threading

CALLBACK_JOBS = 8
callback_results = {}
callback_lock = threading.Lock()
callback_done = threading.Event()


def on_inference_done(outputs, user_arg):
    "Runs on a DX-RT worker thread. Keep it short; copy what must outlive the callback."
    label, probability = top_k(outputs[0], k=1)[0]
    with callback_lock:
        on_worker_thread = threading.current_thread() is not threading.main_thread()
        callback_results[user_arg] = (label, probability, on_worker_thread)
        if len(callback_results) == CALLBACK_JOBS:
            callback_done.set()
    return 0


callback_inputs = [real_tensor.copy() for _ in range(CALLBACK_JOBS)]   # alive until the callbacks finish

with InferenceEngine(str(MODEL_PATH), option) as ie:
    ie.register_callback(on_inference_done)
    start = time.perf_counter()
    callback_job_ids = [ie.run_async([array], user_arg=index) for index, array in enumerate(callback_inputs)]
    if not callback_done.wait(timeout=30):
        raise TimeoutError(f"Only {len(callback_results)} of {CALLBACK_JOBS} callbacks arrived.")
    elapsed_s = time.perf_counter() - start
    ie.register_callback(None)

print(f"Submitted job IDs: {callback_job_ids}")
for user_arg in sorted(callback_results):
    label, probability, on_worker_thread = callback_results[user_arg]
    print(f"  user_arg={user_arg}: {label} {probability:.1%}, ran on a DX-RT worker thread: {on_worker_thread}")
print(f"Throughput       : {CALLBACK_JOBS / elapsed_s:.2f} requests/s")

## 6. Batch API

DX-RT batch execution groups several independent samples and schedules them asynchronously inside the runtime. It does **not** change the compiled model's batch dimension; the DXNN model still normally uses batch size 1. The current API uses `run()` with batch-formatted inputs and explicit output buffers; `run_batch()` is retained only as a deprecated compatibility wrapper.

The nested Python form is:

~~~python
[
    [sample_0_input_0],
    [sample_1_input_0],
    [sample_2_input_0],
]
~~~


In [ ]:
BATCH_SIZE = 4

with InferenceEngine(str(MODEL_PATH)) as ie:
    tensor_info = ie.get_input_tensors_info()[0]
    output_info = ie.get_output_tensors_info()
    batch_inputs = [[make_input(tensor_info, index)] for index in range(BATCH_SIZE)]
    batch_output_buffers = [
        [
            np.empty(info["shape"], dtype=info["dtype"])
            for info in output_info
        ]
        for _ in range(BATCH_SIZE)
    ]

    start = time.perf_counter()
    batch_outputs = ie.run(
        batch_inputs,
        output_buffers=batch_output_buffers,
    )
    elapsed_s = time.perf_counter() - start

print(f"Batch samples      : {len(batch_outputs)}")
print(f"Outputs per sample : {[len(sample) for sample in batch_outputs]}")
print(f"Effective rate     : {BATCH_SIZE / elapsed_s:.2f} samples/s")


## 7. Buffer count is pipeline capacity

<code>InferenceOption.buffer_count</code> controls the number of internal inference buffers. A larger value can permit more in-flight work, but it also consumes more memory and may stop helping after the pipeline is full.

| Too small | Balanced | Too large |
|---|---|---|
| Submitter waits for buffers | Enough work to keep the NPU busy | Extra memory with little gain |
| Lower throughput is possible | Stable throughput and bounded memory | Longer queues can increase latency |

Measure representative values instead of assuming that the largest value is best. The Advanced tutorial performs a controlled buffer-count experiment.


## 8. Build a minimal C++14 application

Since v3.4, DX-RT provides a stable C ABI and a header-only C++14 wrapper. New C++ code should include only:

~~~cpp
#include <dxrt/dxrt_cxx_api.h>
~~~

Do not include <code>dxrt_api.h</code> and <code>dxrt_cxx_api.h</code> in the same translation unit.

DX-RT installs a CMake package configuration (<code>/usr/local/lib/cmake/dxrt/dxrtConfig.cmake</code>), so a project needs only <code>find_package(dxrt REQUIRED)</code> and the imported target <code>dxrt::dxrt</code>. The target carries the include directory and the <code>pthread</code> dependency. The demo tutorials (20-24) wrap the same lookup in a small CMake helper; the lines below are the plain form.

The next two cells create source files only under <code>T06-DX-Runtime/workspace/cpp</code>.


In [ ]:
%%writefile {CPP_DIR / "main.cpp"}
#include <dxrt/dxrt_cxx_api.h>

#include <chrono>
#include <cstdint>
#include <exception>
#include <iostream>
#include <string>
#include <vector>

int main(int argc, char* argv[])
{
    if (argc != 2) {
        std::cerr << "Usage: dxrt_sync <model.dxnn>\n";
        return 2;
    }

    try {
        dxrt::InferenceEngine engine(argv[1]);
        std::vector<std::uint8_t> input(engine.GetInputSize(), 0);

        const auto start = std::chrono::steady_clock::now();
        const auto outputs = engine.Run(input.data());
        const auto end = std::chrono::steady_clock::now();

        const double elapsed_ms =
            std::chrono::duration<double, std::milli>(end - start).count();

        std::cout << "Input bytes   : " << input.size() << '\n';
        std::cout << "Output tensors: " << outputs.size() << '\n';
        for (std::size_t i = 0; i < outputs.size(); ++i) {
            std::cout << "  [" << i << "] " << outputs[i]->name()
                      << ", bytes=" << outputs[i]->size_in_bytes() << '\n';
        }
        std::cout << "Host time     : " << elapsed_ms << " ms\n";
    } catch (const std::exception& error) {
        std::cerr << "DX-RT error: " << error.what() << '\n';
        return 1;
    }
    return 0;
}


In [ ]:
%%writefile {CPP_DIR / "CMakeLists.txt"}
cmake_minimum_required(VERSION 3.16)
project(dxrt_sync LANGUAGES CXX)

# Installed by the DX-RT package: /usr/local/lib/cmake/dxrt/dxrtConfig.cmake
find_package(dxrt REQUIRED)

add_executable(dxrt_sync main.cpp)
target_compile_features(dxrt_sync PRIVATE cxx_std_14)
target_link_libraries(dxrt_sync PRIVATE dxrt::dxrt)


### 8.1 Configure and build

These notebook cells run the same commands you would enter in a terminal:

~~~bash
cmake -S <T06>/workspace/cpp -B <T06>/workspace/cpp/build \
      -DCMAKE_BUILD_TYPE=Release
cmake --build <T06>/workspace/cpp/build --parallel
~~~

The build directory stays inside T06.


In [ ]:
!cmake -S "{CPP_DIR}" -B "{CPP_BUILD_DIR}" -DCMAKE_BUILD_TYPE=Release
!cmake --build "{CPP_BUILD_DIR}" --parallel


### 8.2 Run the C++ application

Equivalent terminal command:

~~~bash
<T06>/workspace/cpp/build/dxrt_sync \
  <T06>/workspace/models/resnet50_224x224.dxnn
~~~

For asynchronous, callback, multi-threaded, multi-input, and profiler programs in C++, start from the SDK examples in `<DX_ALL_SUITE_DIR>/dx-runtime/dx_rt/examples/cpp` (for example `run_async_model`, `run_async_model_thread`, `run_batch_model`, `multi_input_model_inference`).


In [ ]:
!"{CPP_BUILD_DIR / 'dxrt_sync'}" "{MODEL_PATH}"


## 9. Choose an execution style

| Requirement | Recommended starting point | Reason |
|---|---|---|
| Simplest sequential flow | Synchronous <code>run()</code> | Clear ownership and error handling |
| Lowest single-request latency study | Synchronous <code>run()</code> | No intentional queue depth |
| Higher streaming throughput | Async + <code>wait()</code> or callback | Overlaps independent pipeline work |
| Several independent samples at once | Batch API | Groups submission and completion |
| Tight integration and low Python overhead | C++ API | Direct C++14 application control |
| Existing Python pipeline | Python API | Fast integration with NumPy data |

The correct choice depends on the complete application. A higher inference FPS can still produce worse user-visible latency if queues become too deep.


## 10. Troubleshooting

| Symptom | What you see | Likely cause | Fix |
|---|---|---|---|
| `uv` or `cmake` is missing | `[MISSING] uv` or `[MISSING] cmake` | Not installed | `curl -LsSf https://astral.sh/uv/install.sh | sh`; `sudo apt install cmake build-essential` |
| No wheel for this Python | `No dx_engine wheel for cp3XY` | The kernel Python version has no matching wheel under `/usr/share/libdxrt-bin/python` | Create the tutorial `.venv` with a supported Python version |
| `import dx_engine` fails | A `RuntimeError` at import about a version mismatch, or an `ImportError` | Since DX-RT v3.5.0, `import dx_engine` stops when the wheel's major.minor version differs from the installed `libdxrt.so`; the wheel in `workspace/.venv-dxrt` came from an older runtime | Rerun section 2 to install the matching wheel. `DX_ENGINE_SKIP_VERSION_CHECK=1` skips the check; use it only for diagnosis |
| Callbacks never arrive | `TimeoutError: Only N of 8 callbacks arrived` | An exception inside the callback, or the callback was cleared before the jobs finished | Keep the callback short and exception-free; clear it with `register_callback(None)` only after the last job |
| CMake cannot find DX-RT | `Could not find a package configuration file provided by "dxrt"` | DX-RT was installed under a custom prefix, so `dxrtConfig.cmake` is not in a default search path | Add `-DCMAKE_PREFIX_PATH=<prefix>` to the `cmake -S ... -B ...` line |
| Async outputs look wrong | Garbage values after `wait()` | The input buffer was released or reused before `wait()` returned | Keep the input array alive until `wait()` returns |


## 11. Summary

### 11.1 API workflow completed

**Read tensor metadata**  
→ **Allocate exact dtype and shape**  
→ **Run synchronously**  
→ **Classify a real image**  
→ **Submit and wait asynchronously, or complete through a callback**  
→ **Group independent samples**  
→ **Build the same flow in C++**

<img src="assets/dx-rt-execution-modes.svg" style="max-width: 1000px; width: 100%;" alt="DX-RT execution modes">

### 11.2 Execution dashboard

| Mode | Submission | Completion | Best first metric |
|---|---|---|---|
| Sync | One request | Return from <code>run()</code> | Request latency |
| Async + Wait | Several job IDs | <code>wait(job_id)</code> | Throughput and queue delay |
| Callback | Several requests | Runtime callback | Callback cost and throughput |
| Batch | Group of samples | Group of outputs | Effective samples/s |
| C++ | Same runtime concepts | C++ tensors | End-to-end application cost |

### 11.3 Completion checklist

- [ ] Install the wheel matching the Jupyter Python ABI
- [ ] Allocate a contiguous tensor from runtime metadata
- [ ] Measure synchronous host, runtime, and NPU timings
- [ ] Classify a real image and match the result of the DX-APP example
- [ ] Complete asynchronous jobs through a thread-safe callback with `user_arg`
- [ ] Preserve asynchronous input lifetime through <code>wait()</code>
- [ ] Use the batch API without changing model batch size
- [ ] Explain the memory/throughput tradeoff of <code>buffer_count</code>
- [ ] Build and run a C++14 DX-RT application
- [ ] Replace dummy input with product preprocessing and validate outputs

> **Remember:** buffer ownership is part of correctness. Performance tuning comes after shape, dtype, layout, lifetime, and output mapping are verified.

### 11.4 SDK examples

The SDK ships a runnable example for each API in this notebook, which is a good starting point for your own code:

| Language | Directory | Examples |
|---|---|---|
| Python | `<DX_ALL_SUITE_DIR>/dx-runtime/dx_rt/examples/python` | `run_sync_model.py`, `run_async_model_wait.py`, `run_async_model_callback.py`, `run_async_model_thread.py`, `run_batch_model.py`, `run_sync_model_bufcount.py`, `multi_input_model_inference.py` |
| C++ | `<DX_ALL_SUITE_DIR>/dx-runtime/dx_rt/examples/cpp` | `run_sync_model`, `run_async_model_wait`, `run_async_model_thread`, `run_batch_model`, `run_async_model_bufcount`, `multi_input_model_inference` |

### 11.5 Next step

Continue with the **Advanced tutorial** to bind devices and cores, profile individual stages, monitor device health, test memory-loaded and multi-input models, and build reproducible release evidence.
